# SigmanInventorySearcher

# __WORK IN PROGRESS__

## Imports

In [ ]:
# std lib
import sys
import urllib
import logging
from pathlib import Path

# Data wrangling
import pandas as pd

# PubChem querying
from CommercialCompoundSearcher.query import query_CID_from_CAS
from CommercialCompoundSearcher.query import query_SMILES_from_CID

# Database management
from CommercialCompoundSearcher.database import connect_pubchem_cache
from CommercialCompoundSearcher.database import close_pubchem_cache
from CommercialCompoundSearcher.database import get_CID_from_identifier
from CommercialCompoundSearcher.database import upsert_identifier_lookup
from CommercialCompoundSearcher.database import upsert_compound

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)-5s - %(asctime)s] %(message)s',
    datefmt='%m/%d/%Y:%H:%M:%S',  # Correct way to format the date
    handlers=[logging.StreamHandler(sys.stdout)]
)

logger = logging.getLogger(__name__)

## Reading in a spreadsheet of the Sigman Inventory
Export a copy of the Sigman inventory from labsuit and point the inventory_spreadsheet variable to the file.

In [ ]:
# Define the spreadsheet file and the sheet name for the "Chemical" subset of the inventory
inventory_spreadsheet = Path('./data/Sigman-inventory-03-09-2026_small.xlsx')
sheet_name = 'Chemical'

# Define a directory to store the results
results_dir = Path('./results/')
results_dir.mkdir(exist_ok=True, parents=True)

# Define path to SQLite cache
# This stores the results to expedite restarts
db_path = Path('./data/pubchem_cache.sqlite')

# Control how often database commits (lower values are more frequent)
commit_every = 10

# Read in the file
try:
    df = pd.read_excel(inventory_spreadsheet, header=0, sheet_name='Chemical', engine='openpyxl')
except Exception: # Read bytes in case LabSuit is not zipping their .xlsx files
    with open(inventory_spreadsheet, 'rb') as infile:
        df = pd.read_excel(infile, sheet_name='Chemical')

# Replace empty strings with None (converts to NaN) and remove missing CAS numbers
df.replace(to_replace=r'\s+', value=None, inplace=True)
df.dropna(subset='CAS_NUMBER', inplace=True)

display(df)

## Query Pubchem for CID from CAS

The best identifier to use for querying Pubchem is the Pubchem Compound ID (CID). For more information on how Pubchem standardizes its database, please see the [compounds webpage](https://pubchem.ncbi.nlm.nih.gov/docs/compounds). This section will obtain a CID for a given InChI key. The REST queries each take at least 200 ms.

In [ ]:
# Get CAS numbers as a list
cas_numbers = df['CAS_NUMBER'].to_list()

# Initialize CID column as nullable integer
df['CID'] = pd.Series(pd.NA, index=df.index, dtype='Int64')

# Get the total length of CAS numbers for tracking progress
total = len(cas_numbers)

# Open the cache database
conn = connect_pubchem_cache(db_path)

try:
    for i, cas in enumerate(cas_numbers, start=1):

        logger.info('Working on %d of %d (%.2f%%)', i, total, i / total * 100)

        # Skip missing CAS values
        if pd.isna(cas):
            continue

        cas = str(cas).strip()

        # Try cache first using the unified identifier lookup table
        cid, status = get_CID_from_identifier(conn=conn,
                                              identifier_type='cas',
                                              identifier_value=cas)

        # Successful lookup of the CID
        if status == 'ok' and cid is not None:
            df.loc[df['CAS_NUMBER'] == cas, 'CID'] = int(cid)
            continue

        # Validate status. Everything past this block goes to PubChem query
        if status not in ['ok', 'not_found', 'http_error', 'parse_error', None]:
            raise ValueError(f'Unexpected cache status "{status}"')

        # Query PubChem
        try:
            cid = query_CID_from_CAS(cas)
        except urllib.error.HTTPError as e:

            # Cache the failure
            upsert_identifier_lookup(conn=conn,
                                     identifier_type='cas',
                                     identifier_value=cas,
                                     cid=None,
                                     status='http_error')

            logger.error('Could not get CID from CAS %s because', cas)
            logger.error(str(e))
            continue

        # Normalize / handle empty results
        if cid is None:
            upsert_identifier_lookup(conn=conn,
                                     identifier_type='cas',
                                     identifier_value=cas,
                                     cid=None,
                                     status='not_found')

            df.loc[df['CAS_NUMBER'] == cas, 'CID'] = pd.NA

        else:
            cid = int(cid)

            # Ensure parent compound row exists before storing the lookup
            upsert_compound(conn=conn,
                            cid=cid)

            # Cache the CAS to CID mapping
            upsert_identifier_lookup(conn=conn,
                                     identifier_type='cas',
                                     identifier_value=cas,
                                     cid=cid,
                                     status='ok')

            df.loc[df['CAS_NUMBER'] == cas, 'CID'] = cid

        # Commit periodically
        if i % commit_every == 0:
            conn.commit()

    # Final commit
    conn.commit()

finally:
    close_pubchem_cache(conn)

# Show updated DataFrame
display(df)

## Query Pubchem for SMILES from CID

In [ ]:
# Get CAS numbers as a list
cas_numbers = df['CAS_NUMBER'].to_list()

# Initialize output columns
df['CID'] = pd.Series(pd.NA, index=df.index, dtype='Int64')
df['SMILES'] = pd.Series(pd.NA, index=df.index, dtype='string')

# Get the total length of CAS numbers for tracking progress
total = len(cas_numbers)

# Open the cache database
conn = connect_pubchem_cache(db_path)

try:
    for i, cas in enumerate(cas_numbers, start=1):

        logger.info('Working on %d of %d (%.2f%%)', i, total, i / total * 100)

        # Skip missing CAS values
        if pd.isna(cas):
            continue

        cas = str(cas).strip()

        # Try cache first for CAS to CID
        cid, status = get_CID_from_identifier(conn=conn,
                                              identifier_type='cas',
                                              identifier_value=cas)

        # Query PubChem for CID if needed
        if not (status == 'ok' and cid is not None):

            if status not in ['ok', 'not_found', 'http_error', 'parse_error', None]:
                raise ValueError(f'Unexpected cache status "{status}"')

            try:
                cid = query_CID_from_CAS(cas)
            except urllib.error.HTTPError as e:
                upsert_identifier_lookup(conn=conn,
                                         identifier_type='cas',
                                         identifier_value=cas,
                                         cid=None,
                                         status='http_error')

                logger.error('Could not get CID from CAS %s because', cas)
                logger.error(str(e))
                continue

            if cid is None:
                upsert_identifier_lookup(conn=conn,
                                         identifier_type='cas',
                                         identifier_value=cas,
                                         cid=None,
                                         status='not_found')

                df.loc[df['CAS_NUMBER'] == cas, 'CID'] = pd.NA
                df.loc[df['CAS_NUMBER'] == cas, 'SMILES'] = pd.NA
                continue

            cid = int(cid)

            # Ensure parent compound row exists before storing the lookup
            upsert_compound(conn=conn,
                            cid=cid)

            # Cache the CAS to CID mapping
            upsert_identifier_lookup(conn=conn,
                                     identifier_type='cas',
                                     identifier_value=cas,
                                     cid=cid,
                                     status='ok')

        else:
            cid = int(cid)

        # Store CID in the DataFrame
        df.loc[df['CAS_NUMBER'] == cas, 'CID'] = cid

        # Try cache first for SMILES, preferring canonical_smiles
        row = conn.execute(
            '''
            SELECT canonical_smiles, isomeric_smiles
            FROM compounds
            WHERE cid = ?
            ''',
            (cid,)
        ).fetchone()

        smiles = None

        if row is not None:
            if row['canonical_smiles'] is not None:
                smiles = row['canonical_smiles']
            elif row['isomeric_smiles'] is not None:
                smiles = row['isomeric_smiles']

        # Query PubChem for SMILES if needed
        if smiles is None:
            try:
                smiles = query_SMILES_from_CID(cid)
            except urllib.error.HTTPError as e:
                logger.error('Could not get SMILES from CID %d because', cid)
                logger.error(str(e))
                continue

            if smiles is not None:
                upsert_compound(conn=conn,
                                cid=cid,
                                canonical_smiles=str(smiles))

        # Store SMILES in the DataFrame
        if smiles is not None:
            df.loc[df['CAS_NUMBER'] == cas, 'SMILES'] = str(smiles)
        else:
            df.loc[df['CAS_NUMBER'] == cas, 'SMILES'] = pd.NA

        # Commit periodically
        if i % commit_every == 0:
            conn.commit()

    # Final commit
    conn.commit()

finally:
    close_pubchem_cache(conn)

# Show updated DataFrame
display(df)

## Save the results

In [ ]:
df.to_excel(results_dir / './inventory_with_smiles.xlsx',
            sheet_name=sheet_name,
            index=False)